# Gabriel et al. (2025) on dandiset 001887 and the paper's Zenodo record

**Paper.** Gabriel CJ, Gupta T, Sanchez-Fuentes A, Zeidler Z, Wilke SA, DeNardo LA (2025).
Transformations in prefrontal ensemble activity underlying rapid threat avoidance learning.
*Current Biology* 35, 1128-1136. [doi:10.1016/j.cub.2025.01.010](https://doi.org/10.1016/j.cub.2025.01.010).
The claims file was written from the NIH author manuscript in PubMed Central (PMC11916606) with its
STAR Methods; the supplement was not read by the reader.

**Dandiset.** [DANDI:001887](https://dandiarchive.org/dandiset/001887/0.260902.1724), version
0.260902.1724, the only published version (2026-09-02). The paper cites no dandiset version (it
predates the dandiset), so the most recent published version is pinned; the draft shows the same
56 assets and size.

**Zenodo record.** The dandiset holds no calcium imaging and BehaviorDEPOT labels for only two
sessions. The paper's STAR Methods name its "Analyzed datasets" on Zenodo
([doi:10.5281/zenodo.13381210](https://doi.org/10.5281/zenodo.13381210), Day 1.zip and Day 2.zip),
which hold for every imaged session the Minian traces, the BehaviorDEPOT output, the trial results and
the camera and miniscope timestamps. The claims are tested on that record (`data: elsewhere`), streamed
by HTTP range requests on the zip members, and counted apart from the dandiset. The analysis code read is
Zenodo doi:10.5281/zenodo.13351302.

| Claims | Status here |
|---|---|
| zenodo-dandiset-link (added by the author) | tested |
| data-neurons | tested |
| data-registered-cells, fig1o-tone-to-mixed, fig1o-nonmod-to-mixed | blocked: no cell registration is public |
| fig1e, fig1f, fig1g, fig1h (behaviour) | tested |
| fig1kl-overall, fig1kl-day1-categories, disc-tone-cells-baseline, fig1k-mixed-day2 | tested |
| fig1p, fig1q, fig1pq, fig1r, figs1h (mean activity) | tested |
| figs2-innate-behaviors | blocked: no jumping or rearing labels; BehaviorDEPOT is MATLAB |
| fig2-day1, fig2-day2, fig2e, fig2f | tested |
| fig3a (three claims), fig3b, fig3c (decoding) | tested |
| fig4b-hybrid-model-best | tested |
| figs3gh-behavior-similar | tested |
| fig4e to fig4l, figs3ab, figs3cd, figs3ef, figs4ab, figs4cd (CILDS, 13 claims) | blocked: CILDS is MATLAB |

The notebook takes about 30 minutes, 15 of them for the 64 decoders of Figure 3 and 5 for streaming.

In [ ]:
import time
import warnings
import zlib
from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pingouin as pg
import statsmodels.formula.api as smf
from scipy import stats

import pma

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

started = time.time()
meta = pma.dandi_metadata()
assets = pma.dandi_assets()
nwb = pma.read_nwb_all(assets)
print(f"dandiset {pma.DANDISET} version {pma.VERSION}: {len(assets)} assets, "
      f"{len(nwb)} NWB files read in {time.time() - started:.0f} s")
S = pma.zenodo_sessions()
beh = pma.load_parallel(pma.load_behaviour, S)
print(f"Zenodo record {pma.ZENODO_RECORD}: behaviour of {len(beh)} sessions read, "
      f"{time.time() - started:.0f} s")
ca = pma.load_parallel(pma.load_calcium, S)
print(f"calcium traces of {len(ca)} sessions read, {time.time() - started:.0f} s")
CA_T = {s: pma.ca_times(ca[s]) for s in S}
GROUPS = ("Shock", "NonShock")

## Data check

### The dandiset

In [ ]:
print("Dandiset title:", meta["name"])
print("Description:", meta.get("description"))
print("Assets summary:", {k: meta["assetsSummary"].get(k) for k in ("numberOfSubjects", "numberOfFiles", "numberOfBytes", "variableMeasured")})
kinds = pd.Series([a["path"].rsplit(".", 1)[-1] for a in assets]).value_counts()
print("Asset types:", kinds.to_dict())
rows = []
for k, r in sorted(nwb.items()):
    tr = r["trials"]
    rows.append(dict(subject=r["subject"], sex=r["sex"], session=r["session"], schema=r["notes"]["conversion_schema"],
                     source=r["notes"]["source"], start=r["start_time"][:19], kB=r["size"] // 1000,
                     trials=None if tr is None else len(tr["start_time"]),
                     shock_trials=None if tr is None else int(np.sum(tr["shock_present"])),
                     shock_intervals=None if r["shocks"] is None else len(r["shocks"]),
                     video_frames=r["video_frames"], behaviour_series=len(r["processing"]), imaging=r["imaging"]))
inv = pd.DataFrame(rows)
print(inv.to_string(index=False))
print("\nSubjects:", inv.subject.nunique(), "| sessions by name:", inv.session.value_counts().to_dict())
print("Files with any imaging data:", int(inv.imaging.sum()), "| files with BehaviorDEPOT arrays:",
      inv[inv.behaviour_series > 0][["subject", "session"]].values.tolist())

In [ ]:
# Recordings deposited more than once: identical trial tables or the same source file.
dup = inv[inv.source.duplicated(keep=False)]
print("Files converted from the same source file:\n", dup[["subject", "session", "source", "video_frames"]].to_string(index=False))
keys = [k for k, r in nwb.items() if r["trials"] is not None]
for a, b in combinations(keys, 2):
    ta, tb = nwb[a]["trials"]["start_time"], nwb[b]["trials"]["start_time"]
    if len(ta) == len(tb) and np.array_equal(ta, tb):
        print("identical trial start times:", a, b)
ns_shock = inv[(inv.session.str.contains("NoShock")) & (inv.shock_intervals.fillna(0) > 0)]
print("\nNon-shocked sessions that carry shock intervals:\n", ns_shock[["subject", "session", "shock_trials", "shock_intervals", "source"]].to_string(index=False))

The dandiset holds 30 NWB files for 16 mice (9 shocked, 7 non-shocked) and 26 behaviour videos as
external AVI files. No file holds calcium imaging, cell footprints or traces: the description
("Data collected for/analyzed in Gabriel et al (2025)") is not supported for the imaging. Two files
(F1 and F2, day 1) hold the BehaviorDEPOT arrays and no trial table, and their session start time is
2026-08-25, the conversion date. F6 day 1 holds only the video. The other 26 hold a trial table (tone
and shock times) and no behaviour labels. The F7 day 2 file (non-shocked) was converted from M7's day 2
timestamp file: its trial table is identical to M7's and lists 12 shock trials. The F3 day 1 file
(non-shocked) lists 9 shock trials. The F3 day 2 table has 9 trials, the first at 0 s. None of these
files is used for a result below except as the link of the next section.

### The Zenodo record

In [ ]:
zrows = []
for s in S:
    b, c = beh[s], ca[s]
    cam_fps = (len(b["cam_t"]) - 1) / (b["cam_t"][-1] - b["cam_t"][0])
    ca_t = CA_T[s]
    tw = pma.tone_windows(b, "clock")
    r = nwb.get(pma.nwb_key(s))
    tr = None if r is None else r["trials"]
    on = tw[:, 0]
    off = None
    if tr is not None and len(tr["start_time"]) == len(on):
        d = (b["tone_bouts"][:, 0] / pma.CODE_FPS) - tr["start_time"]
        off = (np.nanmin(d), np.nanmax(d))
    zrows.append(dict(day=s[0], group=s[1], mouse=s[2], cells=c["dff"].shape[0], ca_frames=c["dff"].shape[1],
                      ca_fps=round((len(ca_t) - 1) / (ca_t[-1] - ca_t[0]), 2), ca_min=round((ca_t[-1] - ca_t[0]) / 60, 1),
                      cam_frames=len(b["platform"]), cam_ts=len(b["cam_t"]), cam_fps=round(cam_fps, 2),
                      tones=int(np.sum(~np.isnan(on))), bout_frames=int(np.nanmedian(np.diff(b["tone_bouts"], axis=1))),
                      dff_min=round(float(c["dff"].min()), 2), dff_max=round(float(c["dff"].max()), 1),
                      plat_frac=round(b["platform"].mean(), 2), frz_frac=round(b["freezing"].mean(), 2),
                      nwb_trials=None if tr is None else len(tr["start_time"]),
                      offset_s=None if off is None else f"{off[0]:.2f} to {off[1]:.2f}",
                      hand_scored=b["pma"].get("avoids") is not None))
Z = pd.DataFrame(zrows)
print(Z.to_string(index=False))
print("\nSessions per day and group:\n", Z.groupby(["day", "group"]).mouse.apply(lambda x: f"{len(x)}: {', '.join(x)}").to_string())
print("\nCells per day and group:\n", Z.groupby(["day", "group"]).cells.sum().to_string())

The Zenodo record holds 30 sessions: day 1, 9 shocked and 7 non-shocked mice; day 2, 7 and 7 (F1 and
F2 have no day 2). The paper's Figure 1 behaviour legend gives 11 and 9 mice on day 1 and 8 and 8 on day
2: the behaviour-only mice are not deposited. Calcium frames run at 14.9 frames/s (every second
miniscope timestamp, as the 2x temporal downsampling implies) for 28 to 33 minutes, and the miniscope
timestamps number twice the calcium frames in every session. Every session has 12 tones except F3 on
day 2 (10; tones 2 and 3 are NaN in BehaviorDEPOT). The BehaviorDEPOT tone bouts last 751 frames, 30 s at
25 frames/s. The camera timestamps give 25.0 frames/s in 24 sessions but 28.2 to 29.8 frames/s in six
(F5 and M5 on both days, F6 on both days): there the bouts, placed at 25 frames/s, drift from the real
tones by up to several minutes by the end of the session. This is a finding about the paper's
processing; the `clock` tone windows (pma.tone_windows) correct it, and the `paper` windows keep it.

### Clock check

Two contrasts test where the tones and shocks were. First, behaviour: the hand-scored trial
outcomes against the code's rule on each tone reading. Second, calcium: in shocked mice the
population response in the 2 s after the shock onset, against the 2 s before, under each tone
reading and with the shock times moved by 60 s.

In [ ]:
crow = []
for s in S:
    b = beh[s]
    man = b["pma"].get("avoids")
    if man is None:
        continue
    out = dict(day=s[0], group=s[1], mouse=s[2], cam_fps=Z.set_index(["day", "group", "mouse"]).loc[s[:3], "cam_fps"])
    for rd in ("paper", "clock"):
        m = pma.trial_measures_code(b) if rd == "paper" else pma.trial_measures_clock(b)
        ok = ~np.isnan(m["avoid"])
        out[f"rule_vs_hand_{rd}"] = int((m["avoid"][ok] != man).sum())
    crow.append(out)
CK = pd.DataFrame(crow)
CK["fast_camera"] = CK.cam_fps > 26
print("Trials where the code's success rule disagrees with the hand-scored outcome (12 trials per session):")
print(CK.to_string(index=False))
print("\nTotal disagreements by camera rate:\n", CK.groupby("fast_camera")[["rule_vs_hand_paper", "rule_vs_hand_clock"]].sum().to_string())

def shock_response(s, reading, shift=0.0):
    tw = pma.tone_windows(beh[s], reading)
    t = CA_T[s]
    x = ca[s]["dff"]
    z = (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-9)
    pop = z.mean(0)
    d = []
    for off in tw[~np.isnan(tw[:, 0]), 1]:
        sh = off - 2.0 + shift
        pre, post = (t >= sh - 2) & (t < sh), (t >= sh) & (t < sh + 2)
        if pre.any() and post.any():
            d.append(pop[post].mean() - pop[pre].mean())
    return np.mean(d)

rows = []
for s in S:
    if s[1] != "Shock":
        continue
    rows.append(dict(day=s[0], mouse=s[2], cam_fps=Z.set_index(["day", "group", "mouse"]).loc[s[:3], "cam_fps"],
                     clock=shock_response(s, "clock"), paper=shock_response(s, "paper"),
                     clock_shift_60s=shock_response(s, "clock", 60.0), clock_shift_m60s=shock_response(s, "clock", -60.0)))
SR = pd.DataFrame(rows)
print("\nPopulation z-scored DeltaF/F, mean over cells, 2 s after minus 2 s before the shock onset, mean over trials:")
print(SR.round(3).to_string(index=False))

For the six fast-camera sessions the code's success rule disagrees with the hand-scored outcomes on
15 trials with the paper's tone bouts and on 4 with the `clock` windows; for the 25 frames/s sessions
the readings coincide (21 and 20 disagreements, the rule's own error against hand scoring). So the
behaviour labels and the `clock` tone times share one clock, and the paper's bouts do not in the
fast-camera sessions. The calcium contrast does not decide: the population response to the shock is
within plus or minus 0.08 z under every reading, including the 60 s shifts, so a shared clock for the
calcium and the tones is consistent with the timestamps (both come from the Miniscope DAQ software)
but not shown by a response here. All analyses below use the `clock` windows; the paper's windows are
a variant for the behaviour and for some of the ROC results.

## Link between the dandiset and the Zenodo record

In [ ]:
lrows = []
for s in S:
    r = nwb.get(pma.nwb_key(s))
    b = beh[s]
    row = dict(day=s[0], group=s[1], mouse=s[2], in_dandiset=r is not None)
    if r is not None:
        tr = r["trials"]
        row["dandiset_trials"] = None if tr is None else len(tr["start_time"])
        row["zenodo_tones"] = int(np.sum(~np.isnan(b["tone_bouts"][:, 0])))
        if tr is not None and len(tr["start_time"]) == len(b["tone_bouts"]):
            d = b["tone_bouts"][:, 0] / pma.CODE_FPS - tr["start_time"]
            row["onset_offset_spread_s"] = round(float(np.nanmax(d) - np.nanmin(d)), 3)
        else:
            row["onset_offset_spread_s"] = None
        row["video_frames"] = r["video_frames"]
        row["zenodo_cam_frames"] = len(b["platform"])
        if r.get("beh") is not None:
            pv = r["beh"]["Behavior_Platform_Vector"].astype(bool)
            fv = r["beh"]["Behavior_Freezing_Vector"].astype(bool)
            row["dandiset_labels_equal"] = bool(len(pv) == len(b["platform"]) and np.array_equal(pv, b["platform"]) and np.array_equal(fv, b["freezing"]))
    lrows.append(row)
L = pd.DataFrame(lrows)
print(L.to_string(index=False))
matched = L.in_dandiset & (L.onset_offset_spread_s.fillna(99) < 0.1)
print(f"\nZenodo sessions with a dandiset file: {int(L.in_dandiset.sum())} of {len(L)}")
print(f"... whose dandiset trial table gives the Zenodo tone onsets up to one offset (spread < 0.1 s): {int(matched.sum())}")
print("Sessions without such a table:", L[~matched][["day", "group", "mouse"]].values.tolist())
print("Dandiset files with no Zenodo session:", sorted(set(nwb) - {pma.nwb_key(s) for s in S}))

**Result: partially reproduced.** All 30 Zenodo sessions have a dandiset file with the same mouse
and day, and 25 of them have a trial table that gives the Zenodo tone onsets up to one offset per
session. The other five are F1 and F2 day 1 (no trial table), F6 day 1 (video only), F3 day 2 (9 trials
against 10 tones) and F7 day 2 (M7's table). The BehaviorDEPOT arrays of F1 and F2 in the dandiset
equal the Zenodo ones.

## Cell counts

In [ ]:
cc = Z.groupby(["day", "group"]).agg(mice=("mouse", "count"), cells=("cells", "sum")).reset_index()
print(cc.to_string(index=False))
print("Expected: day 1 Shock 1394 cells from 9 mice, NonShock 847 from 7; day 2 Shock 1031, NonShock 967.")

**Result: reproduced.** The Zenodo traces (`ta_dff`, the traces the paper's code uses) hold 1394
cells from 9 shocked mice and 847 from 7 non-shocked mice on day 1, and 1031 and 967 on day 2, the
paper's counts exactly. The dandiset holds none of them.

In [ ]:
names = sorted({n.rsplit("/", 1)[-1] for d in (1, 2) for n in pma.zenodo_listing(d) if not n.endswith("/")})
generic = sorted({("<mouse>_" + n.split("_", 2)[-1]) if n[:1] in "FM" and "_T" in n[:5] else n for n in names})
print("File names in the Zenodo record (mouse prefix replaced):")
for n in generic:
    print("  ", n)
print("Any name mentioning registration or CellReg:", [n for n in names if "reg" in n.lower()])
print("Any registration series in the dandiset:", [k for r in nwb.values() for k in r["processing"] if "reg" in k.lower()])

**Result: blocked (data missing).** Neither the Zenodo record nor the dandiset holds a cell
registration across days (no CellReg output, no file naming a registration), so the 463 and 322
registered cells cannot be counted, and the two Figure 1O claims, which use them, are blocked too.
Redoing the registration would need CellReg, which is MATLAB.

## Behaviour (Figure 1E to 1H)

In [ ]:
TT = {rd: pma.trial_table(S, beh, rd) for rd in ("clock", "paper")}

def block_stats(T, measure, rising=True):
    B = T.groupby(["day", "group", "mouse", "block"])[measure].mean().reset_index()
    res = {}
    for day in (1, 2):
        d = B[B.day == day].dropna()
        full = d.groupby("mouse").block.nunique()
        d = d[d.mouse.isin(full[full == 4].index)]
        aov = pg.mixed_anova(d, dv=measure, within="block", between="group", subject="mouse", correction=True)
        sh = d[d.group == "Shock"].pivot(index="mouse", columns="block", values=measure)
        ns = d[d.group == "NonShock"].pivot(index="mouse", columns="block", values=measure)
        within = {pr: stats.ttest_rel(sh[pr[1]], sh[pr[0]]).pvalue for pr in combinations(range(1, 5), 2)}
        between = {k: stats.ttest_ind(sh[k], ns[k]).pvalue for k in range(1, 5)}
        res[day] = dict(means=d.groupby(["group", "block"])[measure].mean().unstack(), n=d.groupby("group").mouse.nunique().to_dict(),
                        aov=aov, within_all={k: pma.sidak(p, 6) for k, p in within.items()},
                        within_b1={k: pma.sidak(p, 3) for k, p in within.items() if k[0] == 1},
                        between={k: pma.sidak(p, 4) for k, p in between.items()})
    m = res[1]["means"]
    direction = m.loc["Shock", 4] > m.loc["Shock", 1] if rising else m.loc["Shock", 4] < m.loc["Shock", 1]
    verdict = {}
    for fam in ("within_all", "within_b1"):
        w = min(res[1][fam].values()) < 0.05
        btw = min(min(res[d]["between"].values()) for d in (1, 2)) < 0.05
        verdict[fam] = bool(direction and w and btw)
    return res, bool(direction), verdict

def show_behaviour(measure, rising=True):
    out = {}
    for rd in ("clock", "paper"):
        for succ in ("success",):
            res, direction, verdict = block_stats(TT[rd], measure, rising)
            print(f"=== {measure}, tone reading {rd}")
            for day in (1, 2):
                r = res[day]
                print(f"day {day}, mice {r['n']}; block means:")
                print(r["means"].round(3).to_string())
                a = r["aov"].set_index("Source")
                print("  mixed ANOVA: " + "; ".join(f"{src} F={a.loc[src, 'F']:.2f} p={a.loc[src, 'p_unc']:.3g}" + (f" (GG p={a.loc[src, 'p_GG_corr']:.3g})" if 'p_GG_corr' in a and not np.isnan(a.loc[src].get('p_GG_corr', np.nan)) else "") for src in a.index))
                print("  Shock blocks, Sidak over 6 pairs:", {f"{k[0]}-{k[1]}": round(v, 4) for k, v in r["within_all"].items()})
                print("  Shock blocks against block 1, Sidak over 3:", {f"{k[0]}-{k[1]}": round(v, 4) for k, v in r["within_b1"].items()})
                print("  Shock against NonShock per block, Sidak over 4:", {k: round(v, 4) for k, v in r["between"].items()})
            print(f"  day 1 Shock block 4 {'above' if rising else 'below'} block 1: {direction}; criterion met (all pairs / against block 1): {verdict}")
            out[rd] = verdict
    return out

In [ ]:
V_E = show_behaviour("platform")

**Result: reproduced.** On day 1 the shocked mice's fraction of the tone on the platform rises from
0.38 in block 1 to 0.78 in block 4 (`clock`), block 1 against 4 Sidak p = 0.0047 (all pairs) and 0.0024
(against block 1); shocked mice differ from non-shocked mice at day 1 block 4 (p = 0.048) and at day 2
blocks 2 to 4 (p = 0.0046 to 0.017). With the paper's tone bouts the day 1 rise is 0.38 to 0.74 (p =
0.0079 and 0.004) and day 2 blocks 2 to 4 differ (p = 0.0023 to 0.026). Group sizes are 9 and 7 mice on
day 1 and 7 and 7 on day 2, where the paper had 11, 9, 8 and 8.

In [ ]:
V_F = show_behaviour("success")
for rd in ("clock", "paper"):
    T = TT[rd]
    print(f"{rd}: hand-scored sessions {T[T.hand_scored].groupby(['day','group','mouse']).ngroups}; "
          f"rule against hand-scored outcome disagree on {int((T.success != T.success_rule).sum())} of {int(T.success.notna().sum())} trials")

**Result: reproduced.** The proportion of successful trials of shocked mice rises from 0.37 to 0.89
over day 1 (block 1 against 4 Sidak p = 0.0012 over all pairs, 0.0006 against block 1), and shocked mice
are above non-shocked mice at day 2 blocks 2 to 4 (p = 0.0014 to 0.032) under both tone readings.
Success is the hand-scored outcome in the 21 sessions that have one, else the code's rule.

In [ ]:
V_G = show_behaviour("freezing")

**Result: reproduced.** Freezing of shocked mice rises over day 1 from 0.22 to 0.42 (`clock`). Against
block 1 the block 4 difference is Sidak p = 0.047; over all six pairs no difference reaches p < 0.05
(smallest 0.092). With the paper's tone bouts both families pass (block 1 against 4, p = 0.0031 and
0.0015). Shocked mice freeze more than non-shocked mice on day 2 (block 2 p = 0.026 and block 4 p = 0.0012,
`clock`). The criterion names both Sidak families as readings; the claim holds under the block 1 family
with either tone reading and under all pairs with the paper's bouts.

In [ ]:
V_H = show_behaviour("latency", rising=False)

**Result: reproduced, resting on the `clock` tone windows.** With the `clock` windows the shocked
mice's latency falls from 10.8 s in day 1 block 1 to 8.1 s in block 4, the only Sidak-significant block
difference is block 2 against 4 (p = 0.032, all pairs; against block 1 none, smallest 0.14), and the
groups differ at day 2 block 4 (p = 0.043). With the paper's tone bouts no within-group difference
reaches p < 0.05 (smallest 0.33 against block 1, 0.51 over all pairs) although the groups differ at day
2 blocks 2 and 4 (p = 0.047 and 0.037), so the claim fails under that reading. The latency is the
measure most affected by the bout drift in the fast-camera sessions.

In [ ]:
fig, axs = plt.subplots(2, 4, figsize=(15, 6), sharex=True)
for i, meas in enumerate(("platform", "success", "freezing", "latency")):
    B = TT["clock"].groupby(["day", "group", "mouse", "block"])[meas].mean().reset_index()
    for j, day in enumerate((1, 2)):
        ax = axs[j, i]
        for g, col in zip(GROUPS, ("tab:red", "tab:gray")):
            d = B[(B.day == day) & (B.group == g)].groupby("block")[meas]
            ax.errorbar(d.mean().index, d.mean(), d.sem(), color=col, label=g, marker="o")
        ax.set_title(f"{meas}, day {day}")
axs[0, 0].legend()
plt.tight_layout()
plt.show()
for meas in ("platform", "success", "freezing", "latency"):
    v = TT["clock"][meas]
    print(f"{meas}: per-trial values range {v.min():.2f} to {v.max():.2f}; block means plotted, all within the axes")

## ROC encoding (Figure 1I to 1S)

The ROC is run on the `clock` tone windows (the data check), with the calls for each period
under each period reading (P1 tones 9 to 12, P2 tones 10 to 12) and frame reading (F1, F2) of
the claims file, and on the whole session for the mean-activity claims (the code's
session-wide vectors).

In [ ]:
def periods(day):
    if day == 1:
        p = {"All": None, "BL-F1": ((1, 2, 3), "F1"), "BL-F2": ((1, 2, 3), "F2")}
        for pr, tones in (("P1", (9, 10, 11, 12)), ("P2", (10, 11, 12))):
            for fr in ("F1", "F2"):
                p[f"D1-{pr}-{fr}"] = (tones, fr)
        return p
    p = {"All": None}
    for pr, tones in (("P1", (9, 10, 11, 12)), ("P2", (10, 11, 12))):
        for fr in ("F1", "F2"):
            p[f"D2-{pr}-{fr}"] = (tones, fr)
    return p

FEATS = pma.CORE + ["tonePlatentVector", "itiPlatentVector", "tonePlatextVector", "itiPlatextVector"]

def run_roc(reading, which=None):
    out, fe = {}, {}
    for s in S:
        cat = CA_T[s]
        F, ev, tw = pma.features(beh[s], cat, reading)
        fe[s] = (F, ev, tw)
        for pn, pdef in periods(s[0]).items():
            if which is not None and pn not in which:
                continue
            m = np.ones(len(cat), bool) if pdef is None else pma.period_mask(cat, tw, *pdef)
            rng = np.random.default_rng(zlib.crc32(f"{reading}{s[3]}{pn}".encode()))
            feats = FEATS if pn == "All" else pma.CORE
            A = np.full((len(ca[s]["dff"]), len(feats)), np.nan)
            C = np.zeros(A.shape, int)
            for j, fn in enumerate(feats):
                A[:, j], C[:, j] = pma.roc_calls(ca[s]["dff"][:, m], F[fn][m], rng)
            out[(s, pn)] = (feats, A, C)
    return out, fe

t0 = time.time()
ROC, FE = run_roc("clock")
print(f"ROC on {len(ROC)} session periods in {time.time() - t0:.0f} s")
cnt = []
for day in (1, 2):
    for g in GROUPS:
        calls = np.vstack([ROC[(s, "All")][2] for s in S if s[0] == day and s[1] == g])
        cnt.append({"day": day, "group": g, **{f"{n[:-6]} +/-": f"{(calls[:, j] == 1).sum()}/{(calls[:, j] == -1).sum()}" for j, n in enumerate(FEATS)}})
print("Whole-session calls, excited/suppressed cells:")
print(pd.DataFrame(cnt).to_string(index=False))
print("Paper (STAR Methods, Cells Analyzed for Figure 1), excited/suppressed: day 1 ToneOn Shk 61/52 NS 24/25, Frz Shk 162/334 NS 137/187, "
      "PlatEnt Shk 150/87 NS 95/66, PlatExt Shk 131/78 NS 66/67; day 2 ToneOn Shk 34/37 NS 34/22, Frz Shk 171/283 NS 150/198, "
      "PlatEnt Shk 112/80 NS 102/92, PlatExt Shk 104/74 NS 100/82")
ev_n = pd.DataFrame([dict(day=s[0], group=s[1], mouse=s[2], **{k: len(v) for k, v in FE[s][1].items()}) for s in S])
print("\nEvents per session (clock reading):")
print(ev_n.to_string(index=False))

In [ ]:
def cats(day, g, pn, roc=None):
    roc = ROC if roc is None else roc
    return np.vstack([pma.categories(roc[(s, pn)][2][:, :7]) for s in S if s[0] == day and s[1] == g])

def summary(K):
    n = (K != 0).sum(1)
    return dict(cells=len(K), responsive=int((n > 0).sum()), tone_only=int(((K[:, 0] != 0) & (n == 1)).sum()),
                platform_only=int(((K[:, 1] != 0) & (n == 1)).sum()), freezing_only=int(((K[:, 2] != 0) & (n == 1)).sum()),
                mixed=int((n >= 2).sum()), tone_any=int((K[:, 0] != 0).sum()))

READINGS = [(pr, fr) for pr in ("P1", "P2") for fr in ("F1", "F2")]
def pname(per, pr, fr):
    return f"BL-{fr}" if per == "BL" else f"{per}-{pr}-{fr}"
PERDAY = {"BL": 1, "D1": 1, "D2": 2}
SUM = {}
for pr, fr in READINGS:
    for per, day in PERDAY.items():
        for g in GROUPS:
            SUM[(pr, fr, per, g)] = summary(cats(day, g, pname(per, pr, fr)))
SUMT = pd.DataFrame([dict(reading=f"{k[0]}-{k[1]}", period=k[2], group=k[3], **v) for k, v in SUM.items()])
print(SUMT.to_string(index=False))

In [ ]:
rows = []
for pr, fr in READINGS:
    for per in PERDAY:
        a, b = SUM[(pr, fr, per, "Shock")], SUM[(pr, fr, per, "NonShock")]
        c, p = pma.chi2_yates(a["responsive"], a["cells"], b["responsive"], b["cells"])
        rows.append(dict(reading=f"{pr}-{fr}", period=per, shock=a["responsive"] / a["cells"], nonshock=b["responsive"] / b["cells"],
                         diff_pts=100 * (a["responsive"] / a["cells"] - b["responsive"] / b["cells"]), chi2=c, p=p))
OV = pd.DataFrame(rows)
print(OV.round(4).to_string(index=False))
for rd, d in OV.groupby("reading", sort=False):
    sim = (d.p >= 0.05).all()
    inc = all(d.set_index("period").loc["D2", g] > d.set_index("period").loc["BL", g] for g in ("shock", "nonshock"))
    print(f"{rd}: similar in all periods {sim}; D2 above BL in both groups {inc}; both {sim and inc}")

**Result: blocked (methods underspecified).** The fraction of responsive cells rises from BL to D2 in
both groups under all four period and frame readings (shocked 0.46 to 0.52, non-shocked 0.47 to 0.55
under P1-F1). Similarity holds in all three periods only under P2-F2 (smallest p = 0.14); under P1-F1,
P1-F2 and P2-F1 the D1 fractions differ (non-shocked higher by 5.0 to 7.0 points, p = 0.0015 to 0.025).
The claims file makes a claim whose readings disagree methods_underspecified. With the paper's tone
bouts (variant at the end) D1 differs under every reading (p = 0.0003 to 0.047).

In [ ]:
rows = []
for pr, fr in READINGS:
    for per in ("BL", "D1"):
        if per == "BL" and pr == "P2":
            continue
        a, b = SUM[(pr, fr, per, "Shock")], SUM[(pr, fr, per, "NonShock")]
        for k in ("tone_only", "platform_only", "freezing_only", "mixed"):
            c, p = pma.chi2_yates(a[k], a["responsive"], b[k], b["responsive"])
            rows.append(dict(reading=f"{pr}-{fr}" if per == "D1" else fr, period=per, category=k, shock=a[k] / a["responsive"],
                             nonshock=b[k] / b["responsive"], diff_pts=100 * (a[k] / a["responsive"] - b[k] / b["responsive"]), chi2=c, p=p))
CT = pd.DataFrame(rows)
print(CT.round(4).to_string(index=False))
for fr in ("F1", "F2"):
    for pr in ("P1", "P2"):
        d = CT[((CT.period == "BL") & (CT.reading == fr)) | ((CT.period == "D1") & (CT.reading == f"{pr}-{fr}"))]
        print(f"{pr}-{fr}: comparisons with p < 0.05: {int((d.p < 0.05).sum())} of {len(d)}; reproduced {bool((d.p >= 0.05).all())}")

**Result: not reproduced.** Under every period and frame reading at least one category differs between
groups at p < 0.05 by the paper's chi-square test: 2 of 8 comparisons under P1-F1 (BL platform-only,
shocked 24.7% against 33.3%, p = 0.0032; D1 mixed 24.2% against 29.8%, p = 0.043), 1 under P2-F1, 5 under
P1-F2 and 6 under P2-F2 (D1 platform-only 20.2% against 33.8%, p < 0.0001).

In [ ]:
def pooled_area(day, feat, sign, event, half, select=None, roc=None, fe=None):
    roc = ROC if roc is None else roc
    fe = FE if fe is None else fe
    vals = {g: [] for g in GROUPS}
    per_mouse = []
    for s in S:
        if s[0] != day:
            continue
        feats, A, C = roc[(s, "All")]
        sel = (C[:, feats.index(feat)] == sign) if select is None else select(feats, C)
        if sel.sum() == 0:
            per_mouse.append(dict(group=s[1], mouse=s[2], cells=0, events=len(fe[s][1][event]), mean_area=np.nan))
            continue
        area, nev = pma.event_area(ca[s]["dff"][sel], CA_T[s], fe[s][1][event], half)
        vals[s[1]] += list(area[~np.isnan(area)])
        per_mouse.append(dict(group=s[1], mouse=s[2], cells=int(sel.sum()), events=nev, mean_area=np.nanmean(area)))
    a, b = np.array(vals["Shock"]), np.array(vals["NonShock"])
    t = stats.ttest_ind(a, b, equal_var=False)
    return dict(n_shock=len(a), n_ns=len(b), mean_shock=a.mean(), mean_ns=b.mean(), diff=a.mean() - b.mean(), t=t.statistic, p=t.pvalue), pd.DataFrame(per_mouse)

def show_area(label, *args, **kw):
    r, pm = pooled_area(*args, **kw)
    print(f"{label}: cells Shk {r['n_shock']} NS {r['n_ns']}; mean area Shk {r['mean_shock']:.3f} NS {r['mean_ns']:.3f} "
          f"(difference {r['diff']:.3f}); Welch t = {r['t']:.2f}, p = {r['p']:.3g}")
    return r, pm

In [ ]:
rows = []
for pr, fr in READINGS:
    for g in GROUPS:
        f = {per: SUM[(pr, fr, per, g)] for per in PERDAY}
        rows.append(dict(reading=f"{pr}-{fr}", group=g, **{f"G1_{per}": f[per]["tone_any"] / f[per]["cells"] for per in PERDAY},
                         **{f"G2_{per}": f[per]["tone_only"] / f[per]["cells"] for per in PERDAY}))
TB = pd.DataFrame(rows)
print(TB.round(4).to_string(index=False))
for gr in ("G1", "G2"):
    for (rd), d in TB.groupby("reading", sort=False):
        ok = all((r[f"{gr}_BL"] > r[f"{gr}_D1"]) and (r[f"{gr}_BL"] > r[f"{gr}_D2"]) for _, r in d.iterrows())
        print(f"{gr} {rd}: BL greatest in both groups {ok}")
act = []
for day in (1, 2):
    for sign, lab in ((1, "excited"), (-1, "suppressed")):
        r, pm = show_area(f"day {day} tone-onset {lab}", day, "toneonVector", sign, "tone_on", 5.0)
        act.append(r["p"])
print("Activity similar (no p < 0.05):", all(p >= 0.05 for p in act))

**Result: blocked (methods underspecified).** The activity part holds: tone-onset excited and
suppressed cells do not differ between groups on either day (p = 0.18 to 0.58). The proportion part
depends on the readings: under G1 (any tone call) BL is not the greatest in both groups under any
period reading (for example non-shocked P1-F1, BL 16.2% against D2 18.0%); under G2 (tone-only cells) it
is under P1-F1 (shocked 10.7%, 8.8%, 8.9%; non-shocked 9.5%, 7.9%, 9.4%) and not under the other three.
The claims file makes a claim whose readings disagree methods_underspecified.

In [ ]:
rows = []
for pr, fr in READINGS:
    a, b = SUM[(pr, fr, "D2", "Shock")], SUM[(pr, fr, "D2", "NonShock")]
    for den, key in (("R1 responsive", "responsive"), ("R2 all cells", "cells")):
        c, p = pma.chi2_yates(a["mixed"], a[key], b["mixed"], b[key])
        ps, pn = 100 * a["mixed"] / a[key], 100 * b["mixed"] / b[key]
        rows.append(dict(reading=f"{pr}-{fr}", denominator=den, mixed_shock=a["mixed"], n_shock=a[key], pct_shock=ps,
                         mixed_ns=b["mixed"], n_ns=b[key], pct_ns=pn, chi2=c, p=p,
                         within_tol=bool(abs(ps - 51) <= 5.1 and abs(pn - 25) <= 2.5)))
MX = pd.DataFrame(rows)
print(MX.round(3).to_string(index=False))
print("Expected: 51% (tolerance 5.1) against 25% (2.5), chi-square 43.13, p < 0.0001")
print("Reproduced under any reading:", bool((MX.within_tol & (MX.p < 0.05) & (MX.pct_shock > MX.pct_ns)).any()))

**Result: not reproduced.** On day 2 the shocked mice have fewer mixed cells than the non-shocked mice
under every reading: among responsive cells 24.5% against 32.1% (P1-F1, p = 0.007) and 25.0% to 31.3%
against 31.2% to 33.7% under the others; among all cells 12.0% to 16.7% against 15.8% to 18.2%. The
paper reports 51% against 25%. With the paper's tone bouts the shocked share is 25.9% to 32.2% against
28.2% to 33.2% (variant at the end).

In [ ]:
R_P, PM_P = show_area("day 2 platform-exit excited, exits", 2, "platextVector", 1, "plat_ext", 4.0)
print(PM_P.to_string(index=False))
print("Expected cells: 104 Shk (tolerance 10.4), 100 NS (10)")

**Result: partially reproduced.** Day 2 platform-exit excited cells of shocked mice have a larger
post-exit area than those of non-shocked mice (12.09 against 6.43, Welch p = 0.044), but the cells number
77 and 84 against the paper's 104 and 100, outside the tolerance. The result is fragile: the same
analysis with the shuffles of the paper-bout run (variant at the end; platform exits do not depend on
the tone windows) selects 84 and 82 cells and gives p = 0.117.

In [ ]:
R_Q, PM_Q = show_area("day 2 platform-exit suppressed, exits", 2, "platextVector", -1, "plat_ext", 4.0)
print(PM_Q.to_string(index=False))
print("Paper: 74 Shk and 82 NS cells")

**Result: reproduced.** Day 2 platform-exit suppressed cells (40 shocked, 65 non-shocked, against the
paper's 74 and 82) do not differ in post-exit area (-5.83 against -3.64, difference -2.19, Welch p =
0.56).

In [ ]:
fz = []
for sign, lab in ((1, "excited"), (-1, "suppressed")):
    for ev in ("frz_on", "frz_off"):
        r, pm = show_area(f"day 2 freezing-{lab}, {ev}", 2, "freezingVector", sign, ev, 3.0)
        fz.append(r["p"])
print("At least one p < 0.05:", any(p < 0.05 for p in fz))

**Result: reproduced.** Three of the four day 2 freezing comparisons differ at p < 0.05 (172 and 150
excited, 280 and 198 suppressed cells, close to the paper's 171, 150, 283 and 198): excited cells after
freezing onset (7.36 against 10.58, p = 0.035) and offset (-7.38 against -3.91, p = 0.015) are lower in
shocked mice, and suppressed cells after offset are higher (3.57 against 0.39, p = 0.0023).

In [ ]:
def mixed_sel(rule):
    def f(feats, C):
        K = pma.categories(C[:, :7])
        mixed = (K != 0).sum(1) >= 2
        if rule == "M1":
            return mixed & (C[:, feats.index("platextVector")] == 1)
        return mixed & (C[:, :7] == 1).any(1)
    return f
for rule in ("M1", "M2"):
    res = {}
    for day in (1, 2):
        res[day], pm = show_area(f"{rule} day {day} excited mixed cells, exits", day, "platextVector", 1, "plat_ext", 4.0, select=mixed_sel(rule))
    ok = res[1]["diff"] > 0 and res[1]["p"] < 0.05 and res[2]["p"] >= 0.05
    print(f"{rule}: reproduced {ok}")

**Result: reproduced under reading M2, not under M1.** Mixed cells with any excited call (M2, the
code's split) are more active after platform exits in shocked mice on day 1 (250 and 181 cells, -1.21
against -4.77, p = 0.049) and not on day 2 (p = 0.85). Mixed cells with an excited platform-exit call
(M1) differ on neither day (71 and 42 cells, p = 0.49 on day 1; p = 0.27 on day 2). The day 1 effect is
at the threshold.

In [ ]:
R_H, PM_H = show_area("day 2 platform-exit excited, exits with no shock in the next 2 s", 2, "platextVector", 1, "plat_ext_noshock", 4.0)
print("Exits per session followed by a shock within 2 s:", {s[2]: len(FE[s][1]["plat_ext_shock"]) for s in S if s[0] == 2})

**Result: reproduced.** With exits followed by a shock within 2 s left out (at most 2 per session) the
shocked cells remain higher (11.83 against 6.29, Welch p = 0.046). It shares the fragility of
fig1p-platform-exit-excited-day2.

## Platform entries and exits during the tone and the ITI (Figure 2)

In [ ]:
def folded(day, feat):
    rows = []
    for s in S:
        if s[0] != day:
            continue
        feats, A, C = ROC[(s, "All")]
        j = feats.index(feat)
        sel = C[:, j] != 0
        v = np.where(C[sel, j] == 1, A[sel, j], 1 - A[sel, j])
        rows += [dict(group=s[1], mouse=s[2], value=x) for x in v]
    return pd.DataFrame(rows)

def lsd_anova(d):
    """Ordinary two-way ANOVA (group by epoch) and Fisher's LSD from its residual mean square."""
    m = smf.ols("value ~ C(group) * C(epoch)", data=d).fit()
    mse, df = m.mse_resid, m.df_resid
    g = d.groupby(["group", "epoch"]).value
    mean, n = g.mean(), g.count()
    def lsd(a, b):
        t = (mean[a] - mean[b]) / np.sqrt(mse * (1 / n[a] + 1 / n[b]))
        return t, 2 * stats.t.sf(abs(t), df)
    return mean, n, lsd

FIG2 = {}
for day in (1, 2):
    for ev in ("Platent", "Platext"):
        cells = pd.concat([folded(day, f"tone{ev}Vector").assign(epoch="tone"), folded(day, f"iti{ev}Vector").assign(epoch="iti")])
        mice = cells.groupby(["group", "mouse", "epoch"]).value.agg(["mean", "count"]).reset_index()
        mice = mice[mice["count"] >= 3].rename(columns={"mean": "value"})
        for H, d in (("H1", cells), ("H2", mice)):
            mean, n, lsd = lsd_anova(d)
            FIG2[(day, ev, H)] = dict(mean=mean, n=n, lsd=lsd)
            print(f"day {day} {ev} {H}: means " + ", ".join(f"{k[0]} {k[1]} {mean[k]:.3f} (n {n[k]})" for k in mean.index))
            for gname in GROUPS:
                t, p = lsd((gname, "tone"), (gname, "iti"))
                print(f"    {gname} tone vs ITI: t = {t:.2f}, p = {p:.3g}")
            t, p = lsd(("Shock", "tone"), ("NonShock", "tone"))
            print(f"    tone, Shock vs NonShock: t = {t:.2f}, p = {p:.3g}")

In [ ]:
for H in ("H1", "H2"):
    ok = True
    for ev in ("Platent", "Platext"):
        f = FIG2[(1, ev, H)]
        for gname in GROUPS:
            t, p = f["lsd"]((gname, "tone"), (gname, "iti"))
            hit = f["mean"][(gname, "tone")] > f["mean"][(gname, "iti")] and p < 0.05
            ok &= hit
            print(f"{H} day 1 {ev} {gname}: tone {f['mean'][(gname, 'tone')]:.3f} ITI {f['mean'][(gname, 'iti')]:.3f}, LSD p = {p:.3g}, holds {hit}")
    print(f"{H}: reproduced {ok}")

**Result: reproduced.** On day 1 the folded auROC of entry and exit cells is higher for events during
the tone than during the ITI in both groups, pooled over cells (H1, for example shocked entries 0.706
against 0.640, Fisher's LSD p < 1e-30) and over per-mouse scores (H2, p = 2.2e-06 to 0.0017).

In [ ]:
for H in ("H1", "H2"):
    ok = True
    for ev in ("Platent", "Platext"):
        f2, f1 = FIG2[(2, ev, H)], FIG2[(1, ev, H)]
        t, p = f2["lsd"](("Shock", "tone"), ("NonShock", "tone"))
        hit = f2["mean"][("Shock", "tone")] > f2["mean"][("NonShock", "tone")] and p < 0.05
        up = f2["mean"][("Shock", "tone")] > f1["mean"][("Shock", "tone")]
        ok &= hit and up
        print(f"{H} {ev}: day 2 Shock tone {f2['mean'][('Shock', 'tone')]:.3f} NS tone {f2['mean'][('NonShock', 'tone')]:.3f} (LSD p = {p:.3g}); "
              f"Shock day 1 tone {f1['mean'][('Shock', 'tone')]:.3f}; holds {hit and up}")
    print(f"{H}: reproduced {ok}")

**Result: reproduced, resting on reading H1.** Pooled over cells, day 2 shocked tone values exceed
non-shocked ones for entries (0.744 against 0.728, LSD p = 0.034) and exits (0.783 against 0.718, p =
5.4e-13) and exceed shocked day 1 values (0.706 and 0.736). Over per-mouse scores (H2) exits hold (p =
0.0059) and entries do not (0.755 against 0.710, p = 0.086).

In [ ]:
f = FIG2[(2, "Platent", "H2")]
t, p = f["lsd"](("Shock", "tone"), ("NonShock", "tone"))
print(f"day 2 entries, modulation score (mean folded auROC), mice with at least 3 cells: n {f['n'].to_dict()}")
print(f"Shock tone {f['mean'][('Shock', 'tone')]:.3f}, NonShock tone {f['mean'][('NonShock', 'tone')]:.3f}; LSD t = {t:.2f}, p = {p:.3g}")
print("Paper legend: day 2 Shk N=7, NS N=7")

**Result: not reproduced.** The day 2 entry modulation score of shocked mice during the tone is 0.755
against 0.710 for non-shocked mice (LSD p = 0.086), with 6 and 5 mice having at least 3 cells, where the
paper had 7 and 7.

In [ ]:
f = FIG2[(2, "Platext", "H2")]
t, p = f["lsd"](("Shock", "tone"), ("NonShock", "tone"))
print(f"day 2 exits, modulation score, mice with at least 3 cells: n {f['n'].to_dict()}")
print(f"Shock tone {f['mean'][('Shock', 'tone')]:.3f}, NonShock tone {f['mean'][('NonShock', 'tone')]:.3f}; LSD t = {t:.2f}, p = {p:.3g}")
print("Paper legend: day 2 Shk N=6, NS N=7")

**Result: reproduced.** The day 2 exit modulation score during the tone is 0.807 in shocked against
0.705 in non-shocked mice (LSD p = 0.0059), with 4 and 4 mice having at least 3 cells, where the paper
had 6 and 7.

## Decoding trial outcome on day 1 (Figure 3)

Per mouse, day 1 trials 4 to 12, one LIBSVM RBF decoder per 200 ms bin from 20 s before to
50 s after tone onset (a_SVM_permousev3ASF.m, as the claims file gives it). Cell sets for
Figure 3B and 3C come from the ROC calls of the BL period (F1) and the D1 period (P1-F1).

In [ ]:
EPOCHS = {"pre-cue": (-5, 0), "cue-onset": (0, 5), "pre-shock": (23, 28), "shock": (28, 33), "post-shock": (34, 38), "pre-shock-code": (22, 27)}
METHODS_EPOCHS = ["cue-onset", "pre-shock", "shock", "post-shock"]

def cellsets(s):
    kb = pma.categories(ROC[(s, "BL-F1")][2][:, :7])
    kd = pma.categories(ROC[(s, "D1-P1-F1")][2][:, :7])
    bl, d1 = (kb != 0).any(1), (kd != 0).any(1)
    return {"all": np.ones(len(bl), bool), "K1": bl | d1, "K2": d1, "nonmod": ~(bl | d1)}

jobs, keys, DEC_N = [], [], []
for s in S:
    if s[0] != 1:
        continue
    T = TT["clock"]
    lab = T[(T.day == 1) & (T.group == s[1]) & (T.mouse == s[2])].sort_values("trial").success.values[3:12]
    on = pma.tone_windows(beh[s], "clock")[3:12, 0]
    X, BT = pma.decode_inputs(ca[s]["dff"], CA_T[s], on)
    sets = cellsets(s)
    DEC_N.append(dict(group=s[1], mouse=s[2], successes=int(lab.sum()), failures=int((lab == 0).sum()), **{k: int(v.sum()) for k, v in sets.items()}))
    for k, sel in sets.items():
        if sel.sum() > 0:
            jobs.append((X[:, :, sel], lab, zlib.crc32(f"{s[3]}{k}".encode()) % 100000))
            keys.append((s, k, lab))
t0 = time.time()
PRED = dict(zip([(k[0], k[1]) for k in keys], map(pma.decode_job, jobs)))
LAB = {(k[0], k[1]): k[2] for k in keys}
print(f"{len(jobs)} decoders (mouse and cell set), {len(BT)} bins each, in {time.time() - t0:.0f} s")
print(pd.DataFrame(DEC_N).to_string(index=False))

def epoch_table(cellset, positive, norm):
    rows = []
    for (s, k), P in PRED.items():
        if k != cellset:
            continue
        f1 = pma.normalise(pma.f1_scores(P, LAB[(s, k)], positive), norm)
        rows.append(dict(group=s[1], mouse=s[2], **{e: f1[(BT >= a - 1e-9) & (BT < b - 1e-9)].mean() for e, (a, b) in EPOCHS.items()}, trace=f1))
    return pd.DataFrame(rows)

def group_test(E, epochs=METHODS_EPOCHS):
    long = E.melt(id_vars=["group", "mouse"], value_vars=epochs, var_name="epoch", value_name="f1")
    aov = pg.mixed_anova(long, dv="f1", within="epoch", between="group", subject="mouse", correction=True)
    sid = {}
    for e in epochs:
        a, b = E[E.group == "Shock"][e], E[E.group == "NonShock"][e]
        sid[e] = (a.mean(), b.mean(), pma.sidak(stats.ttest_ind(a, b).pvalue, len(epochs)))
    return aov, sid

DEC = {}
for cs in ("all", "K1", "K2", "nonmod"):
    for pos in ("success", "failure"):
        for norm in ("N1", "N2"):
            E = epoch_table(cs, pos, norm)
            aov, sid = group_test(E)
            _, sid_code = group_test(E, ["cue-onset", "pre-shock-code", "shock", "post-shock"])
            DEC[(cs, pos, norm)] = dict(E=E, aov=aov, sid=sid, sid_code=sid_code)
for (cs, pos, norm), d in DEC.items():
    a = d["aov"].set_index("Source")
    print(f"{cs:6s} positive={pos:7s} {norm}: mice Shk {int((d['E'].group == 'Shock').sum())} NS {int((d['E'].group == 'NonShock').sum())}; "
          f"group F={a.loc['group', 'F']:.2f} p={a.loc['group', 'p_unc']:.3g}; "
          + "; ".join(f"{e} {v[0]:.3f}/{v[1]:.3f} Sidak p={v[2]:.3g}" for e, v in d["sid"].items())
          + f"; code pre-shock 22-27 s Sidak p={d['sid_code']['pre-shock-code'][2]:.3g}")

In [ ]:
for pos in ("success", "failure"):
    for norm in ("N1", "N2"):
        d = DEC[("all", pos, norm)]
        ok = all(d["sid"][e][0] > d["sid"][e][1] and d["sid"][e][2] < 0.05 for e in ("pre-shock", "shock"))
        okc = all(d["sid_code"][e][0] > d["sid_code"][e][1] and d["sid_code"][e][2] < 0.05 for e in ("pre-shock-code", "shock"))
        print(f"C{1 if pos == 'success' else 2} {norm}: pre-shock {d['sid']['pre-shock'][0]:.3f} vs {d['sid']['pre-shock'][1]:.3f} (p {d['sid']['pre-shock'][2]:.3g}), "
              f"shock {d['sid']['shock'][0]:.3f} vs {d['sid']['shock'][1]:.3f} (p {d['sid']['shock'][2]:.3g}); holds {ok}; with the code's 22-27 s epoch {okc}")
fig, axs = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, pos in zip(axs, ("failure", "success")):
    E = DEC[("all", pos, "N1")]["E"]
    for g, col in zip(GROUPS, ("tab:red", "tab:gray")):
        tr = np.vstack(E[E.group == g].trace)
        ax.plot(BT, tr.mean(0), color=col, label=g)
    ax.set_title(f"all cells, positive class {pos}, N1")
    ax.set_xlabel("s from tone onset")
axs[0].legend()
plt.show()
print("Plotted: across-mouse means of min-max normalized F1, range 0 to 1 by construction")

**Result: reproduced, resting on the Methods' coding (success positive, min-max normalized F1).**
With success as the positive class and min-max normalization (C1 N1) shocked mice decode better than
non-shocked mice in the pre-shock epoch (0.489 against 0.108, Sidak p = 0.036) and the shock epoch (0.658
against 0.099, p = 0.012), and with the code's 22 to 27 s epoch too. Under the other three readings the
pre-shock difference is not significant (p = 0.39 to 1), nor the shock difference (p = 0.062 to 0.19).
The decoders use 9 shocked and 7 non-shocked mice, where the paper had 7 and 7.

In [ ]:
for pos in ("success", "failure"):
    for norm in ("N1", "N2"):
        E = DEC[("all", pos, norm)]["E"]
        a, b = E[E.group == "Shock"]["pre-cue"], E[E.group == "NonShock"]["pre-cue"]
        p = pma.sidak(stats.ttest_ind(a, b).pvalue, 5)
        print(f"C{1 if pos == 'success' else 2} {norm}: pre-cue (-5 to 0 s) Shk {a.mean():.3f} NS {b.mean():.3f}, Shk higher {a.mean() > b.mean()}; Sidak p (5 epochs) {p:.3g}")

**Result: reproduced, resting on the same reading.** Under C1 N1 shocked mice decode better than
non-shocked mice from 5 s before the tone to its onset (0.481 against 0.134; Sidak p = 0.15, which the
criterion does not use). Under the other three readings shocked mice are not higher (C1 N2 -0.114
against -0.057, C2 N1 0.085 against 0.110, C2 N2 -0.214 against 0.100).

In [ ]:
for pos in ("success", "failure"):
    for norm in ("N1", "N2"):
        E = DEC[("all", pos, norm)]["E"]
        sh = E[E.group == "Shock"]
        means = sh[METHODS_EPOCHS].mean()
        tr = np.vstack(sh.trace).mean(0)
        w = (BT >= 0) & (BT < 38)
        tpk = BT[w][np.argmax(tr[w])]
        print(f"C{1 if pos == 'success' else 2} {norm}: Shock epoch means " + ", ".join(f"{k} {v:.3f}" for k, v in means.items())
              + f"; P1 pre-shock highest {means.idxmax() == 'pre-shock'}; P2 peak bin at {tpk:.1f} s, in 23-28 s {23 <= tpk < 28}")

**Result: not reproduced.** In shocked mice the decoder peaks in the shock epoch under every reading:
the highest bin of the mean trace is at 32.6 s after tone onset, and the shock epoch has the highest
epoch mean (under C1 N1, shock 0.658 against pre-shock 0.489).

In [ ]:
def nsig(d):
    return sum(d["sid"][e][0] > d["sid"][e][1] and d["sid"][e][2] < 0.05 for e in METHODS_EPOCHS)
for cs in ("K1", "K2"):
    for pos in ("success", "failure"):
        for norm in ("N1", "N2"):
            d, a = DEC[(cs, pos, norm)], DEC[("all", pos, norm)]
            em = d["E"].set_index("mouse")
            am = a["E"].set_index("mouse")
            sh = [m for m in em.index if em.loc[m, "group"] == "Shock" and m in am.index]
            e1 = {e: (em.loc[sh, e].mean(), am.loc[sh, e].mean()) for e in ("pre-shock", "shock")}
            E1 = all(v[0] > v[1] for v in e1.values())
            E2 = nsig(d) >= nsig(a)
            print(f"{cs} C{1 if pos == 'success' else 2} {norm}: E1 Shock " + ", ".join(f"{e} {v[0]:.3f} vs all-cell {v[1]:.3f}" for e, v in e1.items())
                  + f" -> {E1}; E2 significant epochs {nsig(d)} vs all-cell {nsig(a)} -> {E2}")

**Result: not reproduced, resting on the Methods' coding.** Under C1 N1, decoding from modulated cells
lowers the shocked mice's pre-shock and shock values (K1: 0.383 and 0.626 against 0.489 and 0.658 with all
cells; K2: 0.442 and 0.582) and leaves no epoch where shocked mice are significantly higher (0 against 3
with all cells). Under the six other readings (C2, or N2) E2 holds in all six and E1 in two (K2 with C2), so
the outcome rests on the C1 N1 reading.

In [ ]:
for pos in ("success", "failure"):
    for norm in ("N1", "N2"):
        d, a = DEC[("nonmod", pos, norm)], DEC[("all", pos, norm)]
        em, am = d["E"].set_index("mouse"), a["E"].set_index("mouse")
        sh = [m for m in em.index if em.loc[m, "group"] == "Shock" and m in am.index]
        e1 = {e: (em.loc[sh, e].mean(), am.loc[sh, e].mean()) for e in ("pre-shock", "shock")}
        E1 = all(v[0] < v[1] for v in e1.values())
        E2 = nsig(d) < nsig(a)
        print(f"nonmod C{1 if pos == 'success' else 2} {norm}: E1 Shock " + ", ".join(f"{e} {v[0]:.3f} vs all-cell {v[1]:.3f}" for e, v in e1.items())
              + f" -> {E1}; E2 significant epochs {nsig(d)} vs all-cell {nsig(a)} -> {E2}")

**Result: blocked (methods underspecified).** Under C1 N1 non-modulated cells give fewer significant
epochs than all cells (0 against 3, E2 holds) but a higher pre-shock value (0.499 against 0.489, E1
fails); the claims file makes a disagreement between E1 and E2 methods_underspecified. Under the other
three readings neither holds.

## Rescorla-Wagner models (Figure 4B)

In [ ]:
def rw_inputs(mouse, trials, seed, rd="clock"):
    T = TT[rd]
    d = T[(T.group == "Shock") & (T.mouse == mouse)].sort_values(["day", "trial"])
    d1 = d[d.day == 1]
    if seed == "S1":
        v0 = d1[d1.trial <= 3].platform.mean()
    else:
        s = [x for x in S if x[0] == 1 and x[1] == "Shock" and x[2] == mouse][0]
        b = beh[s]
        first = pma.tone_windows(b, rd)[0, 0]
        t = b["cam_t"][:len(b["platform"])]
        v0 = b["platform"][:len(t)][t < first].mean()
    if trials == "T1":
        d = d[~((d.day == 1) & (d.trial <= 3))]
    d = d.dropna(subset=["platform", "success"])
    return d.platform.values, d.success.values, v0

RW = {}
for trials in ("T1", "T2"):
    for seed in ("S1", "S2"):
        rows = []
        for mouse in sorted({s[2] for s in S if s[1] == "Shock" and s[0] == 1}):
            data, succ, v0 = rw_inputs(mouse, trials, seed)
            row = dict(mouse=mouse, trials=len(data), v0=v0)
            for model in ("shock", "avoid", "hybrid"):
                a, q, pred = pma.rw_fit(data, succ, v0, model)
                nb = len(data) // 3
                res = [(pred[3 * k:3 * k + 3].mean() - data[3 * k:3 * k + 3].mean()) ** 2 for k in range(nb)]
                row[f"msr_{model}"] = np.mean(res)
                if model == "hybrid":
                    row["alpha"], row["q"] = a, q
                    row["avoid_rate_L1"], row["avoid_rate_L2"] = q, a * q
            rows.append(row)
        D = pd.DataFrame(rows)
        long = D.melt(id_vars="mouse", value_vars=["msr_shock", "msr_avoid", "msr_hybrid"], var_name="model", value_name="msr")
        aov = pg.rm_anova(long, dv="msr", within="model", subject="mouse", correction=True)
        RW[(trials, seed)] = D
        means = D[["msr_shock", "msr_avoid", "msr_hybrid"]].mean()
        pgg = aov["p_GG_corr"].iloc[0] if "p_GG_corr" in aov else aov["p_unc"].iloc[0]
        print(f"=== {trials} {seed}: mice {len(D)}; mean squared residual " + ", ".join(f"{k[4:]} {v:.4f}" for k, v in means.items()))
        print(f"    RM ANOVA F = {aov['F'].iloc[0]:.2f}, p = {aov['p_unc'].iloc[0]:.3g}, GG p = {pgg:.3g}; hybrid lowest {means.idxmin() == 'msr_hybrid'}")
        print(D.round(4).to_string(index=False))

**Result: reproduced.** Under all four readings (T1 and T2, S1 and S2) the hybrid model has the lowest
mean squared residual over blocks (0.0048 to 0.0076, against 0.0243 to 0.0364 for shock-only and 0.0167
to 0.0221 for avoid-only), and the repeated measures ANOVA gives p = 0.0052 to 0.021 (Geisser-Greenhouse
p = 0.012 to 0.050). The fit stops at the starting bound alpha = 1 in every mouse, so the avoid rate q
equals alpha times q. F1 and F2 enter with day 1 only.

## Behaviour in the CILDS epochs (Figure S3G, H)

In [ ]:
rows = []
for s in S:
    if s[0] != 1:
        continue
    b = beh[s]
    T = TT["clock"]
    succ = T[(T.day == 1) & (T.group == s[1]) & (T.mouse == s[2])].set_index("trial").success
    first = next((k for k in range(4, 13) if succ.get(k) == 1), None)
    for ep, (t0, t1) in (("onset", (0, 5)), ("preshock", (23, 28))):
        for which in ("platform", "freezing"):
            v = pma.epoch_fraction(b, "clock", t0, t1, which)
            rows.append(dict(group=s[1], mouse=s[2], epoch=ep, measure=which, B1=np.nanmean(v[3:12]),
                             A2_first_avoid=v[first - 1] if first else np.nan,
                             A1_after_first_avoid=v[first] if first and first < 12 else np.nan, first_avoid=first))
G = pd.DataFrame(rows)
res = []
for (ep, meas), d in G.groupby(["epoch", "measure"]):
    for rd in ("B1", "A2_first_avoid", "A1_after_first_avoid"):
        a, b = d[d.group == "Shock"][rd].dropna(), d[d.group == "NonShock"][rd].dropna()
        t = stats.ttest_ind(a, b, equal_var=False)
        res.append(dict(epoch=ep, measure=meas, reading=rd, n_shock=len(a), n_ns=len(b), shock=a.mean(), nonshock=b.mean(), p=t.pvalue))
GR = pd.DataFrame(res)
print(GR.round(4).to_string(index=False))
for rd, d in GR.groupby("reading"):
    print(f"{rd}: comparisons with p < 0.05: {int((d.p < 0.05).sum())} of {len(d)}")
print("First avoidance trial per mouse:", G.drop_duplicates("mouse")[["group", "mouse", "first_avoid"]].values.tolist())

**Result: reproduced.** Time on the platform and freezing in the tone-onset (0 to 5 s) and pre-shock (23
to 28 s) epochs of day 1 do not differ between groups under any trial reading (12 Welch tests, smallest p
= 0.13; for example platform at onset over trials 4 to 12, 0.63 against 0.42, p = 0.14).

## Variant: the paper's tone labels

The same ROC on the `paper` tone windows, for the claims on proportions and on the mixed
cells of day 2, and the behaviour already shown under both readings above.

In [ ]:
ROCp, FEp = run_roc("paper", which={"All", "BL-F1", "BL-F2", "D1-P1-F1", "D1-P1-F2", "D1-P2-F1", "D1-P2-F2", "D2-P1-F1", "D2-P1-F2", "D2-P2-F1", "D2-P2-F2"})
rows = []
for pr, fr in READINGS:
    for per, day in PERDAY.items():
        a, b = summary(cats(day, "Shock", pname(per, pr, fr), ROCp)), summary(cats(day, "NonShock", pname(per, pr, fr), ROCp))
        c1, p1 = pma.chi2_yates(a["responsive"], a["cells"], b["responsive"], b["cells"])
        c2, p2 = pma.chi2_yates(a["mixed"], a["responsive"], b["mixed"], b["responsive"])
        rows.append(dict(reading=f"{pr}-{fr}", period=per, resp_shock=a["responsive"] / a["cells"], resp_ns=b["responsive"] / b["cells"], p_resp=p1,
                         mixed_of_resp_shock=a["mixed"] / a["responsive"], mixed_of_resp_ns=b["mixed"] / b["responsive"], p_mixed=p2))
print(pd.DataFrame(rows).round(4).to_string(index=False))
r, pm = show_area("paper labels: day 2 platform-exit excited, exits", 2, "platextVector", 1, "plat_ext", 4.0, roc=ROCp, fe=FEp)

In [ ]:
print(f"Notebook run time {(time.time() - started) / 60:.1f} min")

## Summary

| Claim | Outcome |
|---|---|
| zenodo-dandiset-link | partially reproduced |
| data-neurons | reproduced |
| data-registered-cells, fig1o-tone-to-mixed, fig1o-nonmod-to-mixed | blocked: data missing |
| fig1e-platform-time, fig1f-successful-trials, fig1g-freezing | reproduced |
| fig1h-latency | reproduced (assumed: `clock` tone windows) |
| fig1kl-overall-encoding-similar | blocked: methods underspecified |
| fig1kl-day1-categories-similar | not reproduced |
| disc-tone-cells-baseline | blocked: methods underspecified |
| fig1k-mixed-day2 | not reproduced |
| fig1p-platform-exit-excited-day2 | partially reproduced |
| fig1q-platform-exit-inhibited-day2, fig1pq-freezing-day2 | reproduced |
| fig1r-mixed-exit-day1 | reproduced (assumed: reading M2) |
| figs1h-exit-not-shock | reproduced |
| figs2-innate-behaviors | blocked: not runnable |
| fig2-day1-tone-stronger | reproduced |
| fig2-day2-tone-further | reproduced (assumed: reading H1) |
| fig2e-modulation-entries | not reproduced |
| fig2f-modulation-exits | reproduced |
| fig3a-decoding-shock-gt-ns, fig3a-decoding-before-tone | reproduced (assumed: C1 N1) |
| fig3a-decoding-peak-preshock | not reproduced |
| fig3b-feature-cells-enhance | not reproduced (assumed: C1 N1) |
| fig3c-nonmod-cells-worsen | blocked: methods underspecified |
| fig4b-hybrid-model-best | reproduced |
| figs3gh-behavior-similar | reproduced |
| 13 CILDS claims (fig4e to fig4l, figs3ab, figs3cd, figs3ef, figs4ab, figs4cd) | blocked: not runnable |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

How the analysis differs from the paper's: every claim is tested on the Zenodo record, as the dandiset
holds no calcium imaging; the behaviour-only mice of Figure 1 are not deposited (9 and 7 mice on day 1,
7 and 7 on day 2); the tone windows are corrected to the camera clock in six sessions where the camera
ran near 30 frames/s; the ROC routine (caAnalyze), the alignment of calcium and behaviour frames, and
the cell sets of the decoding (ROC_IDs) are not public and were rebuilt from the Methods; the Sidak and
Fisher's LSD comparisons use t-tests corrected for the family, not Prism's pooled errors.

What was found about the dataset: dandiset 001887 holds no imaging, behaviour labels for 2 of 30
sessions, a trial table copied from another mouse (F7 day 2), shock trials in a non-shocked session (F3
day 1) and conversion dates as session start times (F1, F2). The Zenodo record holds the imaged sessions
with the paper's cell counts, and no cell registration.